In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **Libraries**

In [ ]:
!pip install pyvi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 27.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 993.5/993.5 kB 47.3 MB/s eta 0:00:00
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_vendor/pkg_resources/__init__.py", line 3108, in _dep_map
    return self.__dep_map
  File "/usr/local/lib/python3.10/dist-packages/pip/_vendor/pkg_resources/__init__.py", line 2901, in __getattr__
    raise AttributeError(attr)
AttributeError: _DistInfoDistribution__dep_map

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py", line 169, in exc_logging_wrapper
    status = run_func(*args)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/req_command.py", line 242, in wrapper
    return func(self, options, args)
  File "/usr/local/lib/python3.10/dist-packages

In [ ]:
import pandas as pd
import numpy as np
from string import digits
from pyvi import ViTokenizer
from keras.layers import *
from gensim.models.keyedvectors import KeyedVectors
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras import regularizers
from keras.callbacks import EarlyStopping
from keras.models import Model
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
%matplotlib inline

ModuleNotFoundError: ignored

In [ ]:
from tensorflow.keras.models import load_model

## **Data discovery**

In [ ]:
data_train = pd.read_csv("drive/MyDrive/vlsp_sentiment_train.csv", sep='\t')
data_train.columns =['Class', 'Data']

NameError: ignored

In [ ]:
data_train

NameError: ignored

In [ ]:
print(data_train.shape)

Check null and nan values

In [ ]:
# null values
print(data_train.isnull().sum())

In [ ]:

# no of unique classes
print(np.unique(data_train['Class']))

Get labels and reviews

In [ ]:
labels = data_train.iloc[:, 0].values
reviews = data_train.iloc[:, 1].values

In [ ]:
N_CLASS = 3

## **Data preprocessing**

Remove any digits from the reviews, because there is no way we can handle them

In [ ]:
def process_review(reviews: np.ndarray):
    reviews_processed = []
    for review in reviews:
        review_cool_one = ''.join([char for char in review if char not in digits])
        reviews_processed.append(review_cool_one)

    return reviews_processed

reviews_processed = process_review(reviews)

In [ ]:
def oneHotEncode(labels):
    encoded_labels = []
    for label in labels:
        if label == -1:
            encoded_labels.append([1,0,0])
        elif label == 0:
            encoded_labels.append([0,1,0])
        else:
            encoded_labels.append([0,0,1])

    return np.array(encoded_labels)

encoded_labels = oneHotEncode(labels)

print(encoded_labels.shape)

In [ ]:
# Use PyVi for Vietnamese word tokenizer
def tokenize(reviews_processed: list):
    word_reviews = []
    for review in reviews_processed:
        review = ViTokenizer.tokenize(review.lower())
        word_reviews.append(review.split())

    return word_reviews

word_reviews = tokenize(reviews_processed)

Set up some configurations

In [ ]:
EMBEDDING_DIM = 400 # how big is each word vector
MAX_VOCAB_SIZE = 10000 # how many unique words to use (i.e num rows in embedding vector)
MAX_SEQUENCE_LENGTH = 300 # max number of words in a comment to use

Tokenize

In [ ]:
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, lower=True, char_level=False)

In [ ]:
tokenizer.fit_on_texts(word_reviews)
sequences_train = tokenizer.texts_to_sequences(word_reviews)
word_index = tokenizer.word_index

In [ ]:
data = pad_sequences(sequences_train, maxlen=MAX_SEQUENCE_LENGTH)
labels = encoded_labels

In [ ]:
print('Shape of X train and X validation tensor:',data.shape)
print('Shape of label train and validation tensor:', labels.shape)

Create embedding matrix

In [ ]:
def create_embedding():
    # load from pre-trained model

    word_vectors = KeyedVectors.load_word2vec_format('drive/MyDrive/vi-model-CBOW.bin', binary=True)
    vocabulary_size = min(len(word_index)+1,MAX_VOCAB_SIZE)
    embedding_matrix = np.zeros((vocabulary_size, EMBEDDING_DIM))
    print("vocab size", vocabulary_size)
    for word, i in word_index.items():
        if i>=MAX_VOCAB_SIZE:
            continue
        try:
            embedding_vector = word_vectors[word]
            embedding_matrix[i] = embedding_vector
        except KeyError:
            embedding_matrix[i]=np.random.normal(0,np.sqrt(0.25),EMBEDDING_DIM)

    del(word_vectors)

    from keras.layers import Embedding
    embedding_layer = Embedding(vocabulary_size,
                                EMBEDDING_DIM,
                                weights=[embedding_matrix],
                                trainable=True)

    return embedding_layer

embedding_layer = create_embedding()

## **Create model**

### **CNN only**

3 layers

In [ ]:
sequence_length = data.shape[1]
filter_sizes = [3,4,5]
num_filters = 100
drop = 0.5

inputs = Input(shape=(sequence_length,))
embedding = embedding_layer(inputs)

################## CNN ONLY ###############################
reshape = Reshape((sequence_length, EMBEDDING_DIM))(embedding)

conv_0 = Conv1D(num_filters, (filter_sizes[0], ), activation='relu', kernel_regularizer=regularizers.l2(0.01))(reshape)

maxpool_0 = MaxPool1D(296)(conv_0)
conv_1 = Conv1D(num_filters, (filter_sizes[1], ), activation='relu', kernel_regularizer=regularizers.l2(0.01))(reshape)

maxpool_1 = MaxPool1D(296)(conv_1)
conv_2 = Conv1D(num_filters, (filter_sizes[2], ), activation='relu', kernel_regularizer=regularizers.l2(0.01))(reshape)

maxpool_2 = MaxPool1D(296)(conv_2)

print(conv_0.shape, conv_1.shape, conv_2.shape)

maxpool_0 = Reshape((-1, num_filters))(maxpool_0)
maxpool_1 = Reshape((-1, num_filters))(maxpool_1)
maxpool_2 = Reshape((-1, num_filters))(maxpool_2)

print(maxpool_0.shape, maxpool_1.shape, maxpool_2.shape)

concat = concatenate([maxpool_0, maxpool_1, maxpool_2])
flatten = Flatten()(concat)

dropout = Dropout(drop)(flatten)
output = Dense(units=3, activation='softmax', kernel_regularizer=regularizers.l2(0.01))(dropout)

cnn_model = Model(inputs, output)

adam = Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-08, weight_decay=0.0)
cnn_model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])
cnn_model._name = "CNN"
cnn_model.summary()

Use early stopping with delta < 0.01

In [ ]:
#define callbacks
early_stopping = EarlyStopping(monitor='val_loss', min_delta=0.01, patience = 8, verbose=1)
callbacks_list = [early_stopping]

cnn_model.fit(data, labels, validation_split=0.2,
          epochs=20, batch_size=256, callbacks=callbacks_list, shuffle=True)

Plot history

In [ ]:
# Training history
history_dict = cnn_model.history

In [ ]:
# Seperating validation and training accuracy
acc = history_dict.history['accuracy']
val_acc = history_dict.history['val_accuracy']

# Seperating validation and training loss
loss = history_dict.history['loss']
val_loss = history_dict.history['val_loss']

# Plotting
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.plot(acc)
plt.plot(val_acc)
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(['Accuracy', 'Validation Accuracy'])

plt.subplot(1, 2, 2)
plt.plot(loss)
plt.plot(val_loss)
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(['Loss', 'Validation Loss'])

plt.show()

In [ ]:
data_test = pd.read_csv("drive/MyDrive/vlsp_sentiment_test.csv", sep='\t')
data_test.columns =['Class', 'Data']

In [ ]:
data_test

In [ ]:
len(data_test)

In [ ]:
labels_test = data_test.iloc[:, 0].values
reviews_test = data_test.iloc[:, 1].values

In [ ]:
encoded_labels_test = oneHotEncode(labels_test)
reviews_processed_test = process_review(reviews_test)
word_reviews_test = tokenize(reviews_processed_test)
sequences_test = tokenizer.texts_to_sequences(word_reviews_test)
data_test = pad_sequences(sequences_test, maxlen=MAX_SEQUENCE_LENGTH)
labels_test = encoded_labels_test

In [ ]:
print('Shape of X test tensor:',data_test.shape)
print('Shape of label test tensor:', labels_test.shape)

In [ ]:
score = cnn_model.evaluate(data_test, labels_test)

In [ ]:
print("%s: %.2f" % (cnn_model.metrics_names[0], score[0]))
print("%s: %.2f%%" % (cnn_model.metrics_names[1], score[1]*100))

In [ ]:
preds = cnn_model.predict(data_test)
pred_labels = np.argmax(preds, axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix

# Assuming labels_test are the true labels
true_labels = np.argmax(labels_test, axis=1)

# Create confusion matrix
cm = confusion_matrix(true_labels, pred_labels)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,7))
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('Truth')

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

# Calculate metrics
accuracy = accuracy_score(true_labels, pred_labels)
recall = recall_score(true_labels, pred_labels, average='macro')
precision = precision_score(true_labels, pred_labels, average='macro')
f_score = f1_score(true_labels, pred_labels, average='macro')

import pandas as pd

# Create a dictionary with your data
result_sum = {
    'Model': 'CNN',
    'Accuracy': [accuracy],
    'Recall': [recall],
    'Precision': [precision],
    'F-Score': [f_score]
}

# Create a DataFrame from the dictionary
df = pd.DataFrame(result_sum)

# Display the DataFrame
print(df)

### **LSTM only**

In [ ]:
sequence_length = data.shape[1]
drop = 0.5

inputs = Input(shape=(sequence_length,))
embedding = embedding_layer(inputs)

reshape = Reshape((sequence_length,EMBEDDING_DIM))(embedding)

################## LSTM ONLY ###############################
lstm_0 = Bidirectional(LSTM(1024,))(reshape)

dropout = Dropout(drop)(lstm_0)
output = Dense(units=3, activation='softmax', kernel_regularizer=regularizers.l2(0.01))(dropout)
lstm_model = Model(inputs, output)

adam = Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-08, weight_decay=0.0)
lstm_model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])
lstm_model._name = 'LSTM'
lstm_model.summary()

In [ ]:
#define callbacks
early_stopping = EarlyStopping(monitor='val_loss', min_delta=0.01, patience=8, verbose=1)
callbacks_list = [early_stopping]

lstm_model.fit(data, labels, validation_split=0.2,
          epochs=20, batch_size=256, callbacks=callbacks_list, shuffle=True)

In [ ]:
# Training history
history_dict = lstm_model.history

In [ ]:
# Seperating validation and training accuracy
acc = history_dict.history['accuracy']
val_acc = history_dict.history['val_accuracy']

# Seperating validation and training loss
loss = history_dict.history['loss']
val_loss = history_dict.history['val_loss']

# Plotting
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.plot(acc)
plt.plot(val_acc)
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(['Accuracy', 'Validation Accuracy'])

plt.subplot(1, 2, 2)
plt.plot(loss)
plt.plot(val_loss)
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(['Loss', 'Validation Loss'])

plt.show()

In [ ]:
data_test = pd.read_csv("drive/MyDrive/vlsp_sentiment_test.csv", sep='\t')
data_test.columns =['Class', 'Data']

In [ ]:
labels_test = data_test.iloc[:, 0].values
reviews_test = data_test.iloc[:, 1].values

In [ ]:
encoded_labels_test = oneHotEncode(labels_test)
reviews_processed_test = process_review(reviews_test)
word_reviews_test = tokenize(reviews_processed_test)
sequences_test = tokenizer.texts_to_sequences(word_reviews_test)
data_test = pad_sequences(sequences_test, maxlen=MAX_SEQUENCE_LENGTH)
labels_test = encoded_labels_test

In [ ]:
score = lstm_model.evaluate(data_test, labels_test)

In [ ]:
print("%s: %.2f" % (lstm_model.metrics_names[0], score[0]))
print("%s: %.2f%%" % (lstm_model.metrics_names[1], score[1]*100))

In [ ]:
preds = lstm_model.predict(data_test)
pred_labels = np.argmax(preds, axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix

# Assuming labels_test are the true labels
true_labels = np.argmax(labels_test, axis=1)

# Create confusion matrix
cm = confusion_matrix(true_labels, pred_labels)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,7))
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('Truth')

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

# Calculate metrics
accuracy = accuracy_score(true_labels, pred_labels)
recall = recall_score(true_labels, pred_labels, average='macro')
precision = precision_score(true_labels, pred_labels, average='macro')
f_score = f1_score(true_labels, pred_labels, average='macro')

import pandas as pd

# Create a dictionary with your data
result_summary = {
    'Model': 'LSTM',
    'Accuracy': [accuracy],
    'Recall': [recall],
    'Precision': [precision],
    'F-Score': [f_score]
}

# Create a DataFrame from the dictionary
df = pd.DataFrame(result_summary)

# Display the DataFrame
print(df)

### **LSTM + CNN**

In [ ]:
sequence_length = data.shape[1]
filter_sizes = [3,4,5]
num_filters = 100
drop = 0.5

inputs = Input(shape=(sequence_length,))
embedding = embedding_layer(inputs)

################## LSTM ONLY ###############################
reshape = Reshape((sequence_length,EMBEDDING_DIM))(embedding)

################# SINGLE LSTM ####################
# lstm_0 = LSTM(512)(reshape)

# YOU WANNA ADD MORE LSTM LAYERS? UNCOMMENT THIS #
lstm_2 = LSTM(1024, return_sequences=True)(reshape)
lstm_1 = LSTM(512, return_sequences=True)(lstm_2)
lstm_0 = LSTM(256, return_sequences=True)(lstm_1)

############################################################


################## CRNN ####################################
# reshape = Reshape((sequence_length,EMBEDDING_DIM))(embedding)

conv_0 = Conv1D(num_filters, (filter_sizes[0], ),padding="same",activation='relu',kernel_regularizer=regularizers.l2(0.01))(lstm_0)
conv_1 = Conv1D(num_filters, (filter_sizes[1], ),padding="same",activation='relu',kernel_regularizer=regularizers.l2(0.01))(lstm_0)
conv_2 = Conv1D(num_filters, (filter_sizes[2], ),padding="same",activation='relu',kernel_regularizer=regularizers.l2(0.01))(lstm_0)

conv_0 = MaxPool1D(300)(conv_0)
conv_1 = MaxPool1D(300)(conv_1)
conv_2 = MaxPool1D(300)(conv_2)
# Reshape output to match RNN dimension
conv_0 = Reshape((-1, num_filters))(conv_0)
conv_1 = Reshape((-1, num_filters))(conv_1)
conv_2 = Reshape((-1, num_filters))(conv_2)

concat = concatenate([conv_0, conv_1, conv_2])
concat = Flatten()(concat)
# lstm_0 = LSTM(512)(concat)

# YOU WANNA ADD MORE LSTM LAYERS? UNCOMMENT THIS #
# lstm_2 = LSTM(1024, return_sequences=True)(concat)
# lstm_1 = LSTM(512, return_sequences=True)(lstm_2)
# lstm_0 = LSTM(256)(lstm_1)

############################################################

dropout = Dropout(drop)(concat)
output = Dense(units=3, activation='softmax',kernel_regularizer=regularizers.l2(0.01))(dropout)

# this creates a model that includes
lstm_cnn_model = Model(inputs, output)
adam = Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-08, weight_decay=0.0)

lstm_cnn_model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])
lstm_cnn_model._name = 'RCNN'
lstm_cnn_model.summary()

In [ ]:
#define callbacks
early_stopping = EarlyStopping(monitor='val_loss', min_delta=0.01, patience=8, verbose=1)
callbacks_list = [early_stopping]

lstm_cnn_model.fit(data, labels, validation_split=0.2,
          epochs=20, batch_size=256, callbacks=callbacks_list, shuffle=True)

In [ ]:
# Training history
history_dict = lstm_cnn_model.history

In [ ]:
# Seperating validation and training accuracy
acc = history_dict.history['accuracy']
val_acc = history_dict.history['val_accuracy']

# Seperating validation and training loss
loss = history_dict.history['loss']
val_loss = history_dict.history['val_loss']

# Plotting
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.plot(acc)
plt.plot(val_acc)
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(['Accuracy', 'Validation Accuracy'])

plt.subplot(1, 2, 2)
plt.plot(loss)
plt.plot(val_loss)
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(['Loss', 'Validation Loss'])

plt.show()

In [ ]:
data_test = pd.read_csv("drive/MyDrive/vlsp_sentiment_test.csv", sep='\t')
data_test.columns =['Class', 'Data']

In [ ]:
labels_test = data_test.iloc[:, 0].values
reviews_test = data_test.iloc[:, 1].values

In [ ]:
encoded_labels_test = oneHotEncode(labels_test)
reviews_processed_test = process_review(reviews_test)
word_reviews_test = tokenize(reviews_processed_test)
sequences_test = tokenizer.texts_to_sequences(word_reviews_test)
data_test = pad_sequences(sequences_test, maxlen=MAX_SEQUENCE_LENGTH)
labels_test = encoded_labels_test

In [ ]:
score = lstm_cnn_model.evaluate(data_test, labels_test)

In [ ]:
print("%s: %.2f" % (lstm_cnn_model.metrics_names[0], score[0]))
print("%s: %.2f%%" % (lstm_cnn_model.metrics_names[1], score[1]*100))

In [ ]:
preds = lstm_cnn_model.predict(data_test)
pred_labels = np.argmax(preds, axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix

# Assuming labels_test are the true labels
true_labels = np.argmax(labels_test, axis=1)

# Create confusion matrix
cm = confusion_matrix(true_labels, pred_labels)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,7))
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('Truth')

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

# Calculate metrics
accuracy = accuracy_score(true_labels, pred_labels)
recall = recall_score(true_labels, pred_labels, average='macro')
precision = precision_score(true_labels, pred_labels, average='macro')
f_score = f1_score(true_labels, pred_labels, average='macro')

import pandas as pd

# Create a dictionary with your data
result_summary = {
    'Model': 'RCNN',
    'Accuracy': [accuracy],
    'Recall': [recall],
    'Precision': [precision],
    'F-Score': [f_score]
}

# Create a DataFrame from the dictionary
df = pd.DataFrame(result_summary)

# Display the DataFrame
print(df)

### **CNN + LSTM**

In [ ]:
sequence_length = data.shape[1]
filter_sizes = [3,4,5]
num_filters = 100
drop = 0.5

inputs = Input(shape=(sequence_length,))
embedding = embedding_layer(inputs)

################## LSTM ONLY ###############################
reshape = Reshape((sequence_length,EMBEDDING_DIM))(embedding)

################# SINGLE LSTM ####################
# lstm_0 = LSTM(512)(reshape)

# YOU WANNA ADD MORE LSTM LAYERS? UNCOMMENT THIS #
# lstm_2 = LSTM(1024, return_sequences=True)(reshape)
# lstm_1 = LSTM(512, return_sequences=True)(lstm_2)
# lstm_0 = LSTM(256, return_sequences=True)(lstm_1)

############################################################


################## CRNN ####################################

conv_0 = Conv1D(num_filters, (filter_sizes[0], ),padding="same",activation='relu',kernel_regularizer=regularizers.l2(0.01))(reshape)
conv_1 = Conv1D(num_filters, (filter_sizes[1], ),padding="same",activation='relu',kernel_regularizer=regularizers.l2(0.01))(reshape)
conv_2 = Conv1D(num_filters, (filter_sizes[2], ),padding="same",activation='relu',kernel_regularizer=regularizers.l2(0.01))(reshape)

conv_0 = MaxPool1D(300)(conv_0)
conv_1 = MaxPool1D(300)(conv_1)
conv_2 = MaxPool1D(300)(conv_2)
# Reshape output to match RNN dimension
conv_0 = Reshape((-1, num_filters))(conv_0)
conv_1 = Reshape((-1, num_filters))(conv_1)
conv_2 = Reshape((-1, num_filters))(conv_2)

concat = concatenate([conv_0, conv_1, conv_2])
# concat = Flatten()(concat)
# lstm_0 = LSTM(512)(concat)

# YOU WANNA ADD MORE LSTM LAYERS? UNCOMMENT THIS #
lstm_2 = LSTM(1024, return_sequences=True)(concat)
lstm_1 = LSTM(512, return_sequences=True)(lstm_2)
lstm_0 = LSTM(256)(lstm_1)

############################################################

dropout = Dropout(drop)(lstm_0)
output = Dense(units=3, activation='softmax',kernel_regularizer=regularizers.l2(0.01))(dropout)

# this creates a model that includes
cnn_lstm_model = Model(inputs, output)
adam = Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-08, weight_decay=0.0)

cnn_lstm_model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])
cnn_lstm_model._name = 'CRNN'
cnn_lstm_model.summary()

In [ ]:
#define callbacks
early_stopping = EarlyStopping(monitor='val_loss', min_delta=0.01, patience=8, verbose=1)
callbacks_list = [early_stopping]

cnn_lstm_model.fit(data, labels, validation_split=0.2,
          epochs=20, batch_size=256, callbacks=callbacks_list, shuffle=True)

In [ ]:
# Training history
history_dict = cnn_lstm_model.history

In [ ]:
# Seperating validation and training accuracy
acc = history_dict.history['accuracy']
val_acc = history_dict.history['val_accuracy']

# Seperating validation and training loss
loss = history_dict.history['loss']
val_loss = history_dict.history['val_loss']

# Plotting
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.plot(acc)
plt.plot(val_acc)
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(['Accuracy', 'Validation Accuracy'])

plt.subplot(1, 2, 2)
plt.plot(loss)
plt.plot(val_loss)
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(['Loss', 'Validation Loss'])

plt.show()

In [ ]:
data_test = pd.read_csv("drive/MyDrive/vlsp_sentiment_test.csv", sep='\t')
data_test.columns =['Class', 'Data']

In [ ]:
labels_test = data_test.iloc[:, 0].values
reviews_test = data_test.iloc[:, 1].values

In [ ]:
encoded_labels_test = oneHotEncode(labels_test)
reviews_processed_test = process_review(reviews_test)
word_reviews_test = tokenize(reviews_processed_test)
sequences_test = tokenizer.texts_to_sequences(word_reviews_test)
data_test = pad_sequences(sequences_test, maxlen=MAX_SEQUENCE_LENGTH)
labels_test = encoded_labels_test

In [ ]:
score = cnn_lstm_model.evaluate(data_test, labels_test)

In [ ]:
print("%s: %.2f" % (cnn_lstm_model.metrics_names[0], score[0]))
print("%s: %.2f%%" % (cnn_lstm_model.metrics_names[1], score[1]*100))

In [ ]:
preds = cnn_lstm_model.predict(data_test)
pred_labels = np.argmax(preds, axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix

# Assuming labels_test are the true labels
true_labels = np.argmax(labels_test, axis=1)

# Create confusion matrix
cm = confusion_matrix(true_labels, pred_labels)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,7))
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('Truth')

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

# Calculate metrics
accuracy = accuracy_score(true_labels, pred_labels)
recall = recall_score(true_labels, pred_labels, average='macro')
precision = precision_score(true_labels, pred_labels, average='macro')
f_score = f1_score(true_labels, pred_labels, average='macro')

import pandas as pd

# Create a dictionary with your data
result_summary = {
    'Model': 'CRNN',
    'Accuracy': [accuracy],
    'Recall': [recall],
    'Precision': [precision],
    'F-Score': [f_score]
}

# Create a DataFrame from the dictionary
df = pd.DataFrame(result_summary)

# Display the DataFrame
print(df)

### **K-fold with phoBERT**

In [ ]:
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from gensim.utils import simple_preprocess
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

from transformers import get_linear_schedule_with_warmup, AutoTokenizer, AutoModel, logging

import warnings
warnings.filterwarnings("ignore")

logging.set_verbosity_error()

In [ ]:
def seed_everything(seed_value):
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

seed_everything(86)

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
EPOCHS = 6
N_SPLITS = 5


print(device)

In [ ]:
train_df = data_train
test_df = pd.read_csv('drive/MyDrive/vlsp_sentiment_test.csv', sep='\t')

sns.countplot(x='Class', data=train_df)

In [ ]:
train_df = data_train
test_df = pd.read_csv('drive/MyDrive/vlsp_sentiment_test.csv', sep='\t')

train_df['Class'] = train_df['Class'].map({-1:0, 0:1, 1:2})
test_df['Class'] = test_df['Class'].map({-1:0, 0:1, 1:2})


In [ ]:
# We will use Kfold later
# train_df = pd.concat([train_df, valid_df], ignore_index=True)
skf = StratifiedKFold(n_splits=N_SPLITS)
for fold, (_, val_) in enumerate(skf.split(X=train_df, y=train_df.Class)):
    train_df.loc[val_, "kfold"] = fold

In [ ]:
train_df.sample(5)

In [ ]:
train_df.info(), test_df.info()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=120):
        self.df = df
        self.max_len = max_len
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        """
        To customize dataset, inherit from Dataset class and implement
        __len__ & __getitem__
        __getitem__ should return
            data:
                input_ids
                attention_masks
                text
                targets
        """
        row = self.df.iloc[index]
        text, label = self.get_input_data(row)

        # Encode_plus will:
        # (1) split text into token
        # (2) Add the '[CLS]' and '[SEP]' token to the start and end
        # (3) Truncate/Pad sentence to max length
        # (4) Map token to their IDS
        # (5) Create attention mask
        # (6) Return a dictionary of outputs
        encoding = self.tokenizer.encode_plus(
            text,
            truncation=True,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_attention_mask=True,
            return_token_type_ids=False,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_masks': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(label, dtype=torch.long),
        }

    def get_input_data(self, row):
        # Preprocessing: {remove icon, special character, lower}
        text = row['Data']
        text = ' '.join(simple_preprocess(text))
        label = row['Class']

        return text, label

In [ ]:
# Distribution of length of Sentence
all_data = train_df.Data.tolist() + test_df.Data.tolist()
all_data = [' '.join(simple_preprocess(text)) for text in all_data]
encoded_text = [tokenizer.encode(text, add_special_tokens=True) for text in all_data]
token_lens = [len(text) for text in encoded_text]
sns.displot(token_lens)
plt.xlim([0,max(token_lens)])
plt.xlabel('Token Count')

In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, n_classes):
        super(SentimentClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained("vinai/phobert-base")
        self.drop = nn.Dropout(p=0.3)
        self.fc = nn.Linear(self.bert.config.hidden_size, n_classes)
        nn.init.normal_(self.fc.weight, std=0.02)
        nn.init.normal_(self.fc.bias, 0)

    def forward(self, input_ids, attention_mask):
        last_hidden_state, output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=False # Dropout will errors if without this
        )

        x = self.drop(output)
        x = self.fc(x)
        return x

In [ ]:
def train(model, criterion, optimizer, train_loader):
    model.train()
    losses = []
    correct = 0

    for data in train_loader:
        input_ids = data['input_ids'].to(device)
        attention_mask = data['attention_masks'].to(device)
        targets = data['targets'].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(outputs, targets)
        _, pred = torch.max(outputs, dim=1)

        correct += torch.sum(pred == targets)
        losses.append(loss.item())
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        lr_scheduler.step()

    print(f'Train Accuracy: {correct.double()/len(train_loader.dataset)} Loss: {np.mean(losses)}')

def eval(test_data = False):
    model.eval()
    losses = []
    correct = 0

    with torch.no_grad():
        data_loader = test_loader if test_data else valid_loader
        for data in data_loader:
            input_ids = data['input_ids'].to(device)
            attention_mask = data['attention_masks'].to(device)
            targets = data['targets'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            _, pred = torch.max(outputs, dim=1)

            loss = criterion(outputs, targets)
            correct += torch.sum(pred == targets)
            losses.append(loss.item())

    if test_data:
        print(f'Test Accuracy: {correct.double()/len(test_loader.dataset)} Loss: {np.mean(losses)}')
        return correct.double()/len(test_loader.dataset)
    else:
        print(f'Valid Accuracy: {correct.double()/len(valid_loader.dataset)} Loss: {np.mean(losses)}')
        return correct.double()/len(valid_loader.dataset)


In [ ]:
def prepare_loaders(df, fold):
    df_train = df[df.kfold != fold].reset_index(drop=True)
    df_valid = df[df.kfold == fold].reset_index(drop=True)

    train_dataset = SentimentDataset(df_train, tokenizer, max_len=200)
    valid_dataset = SentimentDataset(df_valid, tokenizer, max_len=200)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=5)
    valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=True, num_workers=5)

    return train_loader, valid_loader

In [ ]:
for fold in range(skf.n_splits):
    print(f'-----------Fold: {fold+1} ------------------')
    train_loader, valid_loader = prepare_loaders(train_df, fold=fold)
    model = SentimentClassifier(n_classes=3).to(device)
    criterion = nn.CrossEntropyLoss()
    # Recommendation by BERT: lr: 5e-5, 2e-5, 3e-5
    # Batchsize: 16, 32
    optimizer = AdamW(model.parameters(), lr=2e-5)

    lr_scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=0,
                num_training_steps=len(train_loader)*EPOCHS
            )
    best_acc = 0
    for epoch in range(EPOCHS):
        print(f'Epoch {epoch+1}/{EPOCHS}')
        print('-'*30)

        train(model, criterion, optimizer, train_loader)
        val_acc = eval()

        if val_acc > best_acc:
            torch.save(model.state_dict(), f'phobert_fold{fold+1}.pth')
            best_acc = val_acc

**Test**

In [ ]:
def test(data_loader):
    models = []
    for fold in range(skf.n_splits):
        model = SentimentClassifier(n_classes=3)
        model.to(device)
        model.load_state_dict(torch.load(f'phobert_fold{fold+1}.pth'))
        model.eval()
        models.append(model)

    texts = []
    predicts = []
    predict_probs = []
    real_values = []

    for data in data_loader:
        text = data['text']
        input_ids = data['input_ids'].to(device)
        attention_mask = data['attention_masks'].to(device)
        targets = data['targets'].to(device)

        total_outs = []
        for model in models:
            with torch.no_grad():
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )
                total_outs.append(outputs)

        total_outs = torch.stack(total_outs)
        _, pred = torch.max(total_outs.mean(0), dim=1)
        texts.extend(text)
        predicts.extend(pred)
        predict_probs.extend(total_outs.mean(0))
        real_values.extend(targets)

    predicts = torch.stack(predicts).cpu()
    predict_probs = torch.stack(predict_probs).cpu()
    real_values = torch.stack(real_values).cpu()
    print(classification_report(real_values, predicts))
    return real_values, predicts

In [ ]:
test_dataset = SentimentDataset(test_df, tokenizer, max_len=50)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, num_workers=2)
real_values, predicts = test(test_loader)

In [ ]:
class_names = [0,1,2]
sns.heatmap(confusion_matrix(real_values, predicts), annot=True, xticklabels = class_names, yticklabels = class_names)

In [ ]:
def check_wrong(real_values, predicts):
    wrong_arr = []
    wrong_label = []
    for i in range(len(predicts)):
        if predicts[i] != real_values[i]:
            wrong_arr.append(i)
            wrong_label.append(predicts[i])
    return wrong_arr, wrong_label

for i in range(15):
    print('-'*50)
    wrong_arr, wrong_label = check_wrong(real_values, predicts)
    print(test_df.iloc[wrong_arr[i]].Data)
    print(f'Predicted: ({class_names[wrong_label[i]]}) --vs-- Real label: ({class_names[real_values[wrong_arr[i]]]})')

**Inference**

In [ ]:
def infer(text, tokenizer, max_len=120):
    encoded_review = tokenizer.encode_plus(
        text,
        max_length=max_len,
        truncation=True,
        add_special_tokens=True,
        padding='max_length',
        return_attention_mask=True,
        return_token_type_ids=False,
        return_tensors='pt',
    )

    input_ids = encoded_review['input_ids'].to(device)
    attention_mask = encoded_review['attention_mask'].to(device)

    output = model(input_ids, attention_mask)
    _, y_pred = torch.max(output, dim=1)

    print(f'Text: {text}')
    print(f'Sentiment: {class_names[y_pred]}')

### **Roberta LSTM**

In [ ]:
! pip install evaluate
! pip install py_vncorenlp
# ! pip install -U git+https://github.com/huggingface/transformers.git
! pip install -U git+https://github.com/huggingface/accelerate.git

  Cloning https://github.com/huggingface/accelerate.git to /tmp/pip-req-build-djg5ph3x
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/accelerate.git /tmp/pip-req-build-djg5ph3x
  Resolved https://github.com/huggingface/accelerate.git to commit 244122c736141b164242084c659b6dafa4208fea
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import py_vncorenlp
!mkdir model
py_vncorenlp.download_model(save_dir="model/")

mkdir: cannot create directory ‘model’: File exists
VnCoreNLP model folder model already exists! Please load VnCoreNLP from this folder!


In [ ]:
!mkdir dataset
!mkdir dataset/training
!mkdir output
!mkdir output/checkpoint
!mkdir output/logging

mkdir: cannot create directory ‘dataset’: File exists
mkdir: cannot create directory ‘dataset/training’: File exists
mkdir: cannot create directory ‘output’: File exists
mkdir: cannot create directory ‘output/checkpoint’: File exists
mkdir: cannot create directory ‘output/logging’: File exists


In [ ]:
import os

CURRENT_PATH = os.getcwd()
print(CURRENT_PATH)

/content


In [ ]:
TRAINING_DATASET_PATH = f"{CURRENT_PATH}/dataset/"
OUTPUT_DIR = f"{CURRENT_PATH}/output"
RAW_DATASET_PATH = f"{CURRENT_PATH}/drive/MyDrive/vlsp_sentiment_train.csv"

In [ ]:
import pandas as pd
dataset = pd.read_csv(RAW_DATASET_PATH, sep='\t')

In [ ]:
dataset

,Class,Data
0,-1,"Mình đã dùng anywhere thế hệ đầu, quả là đầy t..."
1,-1,"Quan tâm nhất là độ trễ có cao không, dùng thi..."
2,-1,"dag xài con cùi bắp 98k....pin trâu, mỗi tội đ..."
3,-1,logitech chắc hàng phải tiền triệu trở lên dùn...
4,-1,"Đang xài con m175 cùi mía , nhà xài nhiều chuộ..."
...,...,...
5095,0,Mình mua máy về đc 1 ngày mà điện thoại khác g...
5096,0,Có bạn nào dùng f1w ko.mình dùng m cảm thấy qu...
5097,0,Dùng oppo mà bộ nhớ 4gb thì k chơi games ...
5098,0,Sao tui thích xài hàng oppo mà lựa toàn mấy đứ...


In [ ]:
dataset=dataset.drop_duplicates("Data")
dataset = dataset.dropna()

In [ ]:
import regex as re
import string
import json

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'([a-z]+?)\1+',r'\1', text)
    text = re.sub(r"(\w)\s*([" + string.punctuation + "])\s*(\w)", r"\1 \2 \3", text)
    text = re.sub(r"(\w)\s*([" + string.punctuation + "])", r"\1 \2", text)
    # text = re.sub(r"(\d)([^\d.])", r"\1 \2", text)
    # text = re.sub(r"([^\d.])(\d)", r"\1 \2", text)
    text = re.sub(f"([{string.punctuation}])([{string.punctuation}])+",r"\1", text)
    # Remove numbers
    text = re.sub(r"\d", "", text)
    text = text.strip()
    while text.endswith(tuple(string.punctuation+string.whitespace)):
        text = text[:-1]
    while text.startswith(tuple(string.punctuation+string.whitespace)):
        text = text[1:]
    text = re.sub(r"\s+", " ", text)
    return text

def map_label(label):
    label_map = {
        -1: 0,
        0: 1,
        1: 2
    }
    return label_map[label]

In [ ]:
dataset["Data"] = dataset["Data"].map(lambda text: clean_text(text))
dataset["Class"] = dataset["Class"].map(lambda label: map_label(label))

In [ ]:
dataset

,Class,Data
0,0,"mình đã dùng anywhere thế hệ đầu , quả là đầy ..."
1,0,"quan tâm nhất là độ trễ có cao không , dùng th..."
2,0,"dag xài con cùi bắp k .pin trâu , mỗi tội đánh..."
3,0,logitech chắc hàng phải tiền triệu trở lên dùn...
4,0,"đang xài con m cùi mía , nhà xài nhiều chuột n..."
...,...,...
5095,1,mình mua máy về đc ngày mà điện thoại khác gọi...
5096,1,có bạn nào dùng fw ko . mình dùng m cảm thấy q...
5097,1,dùng opo mà bộ nhớ gb thì k chơi games đc...
5098,1,sao tui thích xài hàng opo mà lựa toàn mấy đứa...


In [ ]:
dataset["Data"].sample(40)

3468    phê quá dành cho doanh nghiệp với ai cần lưu g...
2065          cấu hình ngon , mua về rồi lên vỏ cũng được
2631                                              đẹp ghê
4836            tr là hàng tq nhái bạn ơi . làm gì rẻ vậy
2500                 đang xài surface pro . quá tuyệt vời
4957    thông báo thiết bị thôi mà , mà lần nào sony c...
3229    từ ngày dùng xiaomi mi tôi rất thích xiaomi bở...
4102    hệ điều hành android ngốn nhiều ram quá , máy ...
4028    mình nghĩ để xem phim cũng khá đã đó . thánh n...
4437    thật ra là giống s edge màn hình to có thêm ch...
1350    từ khi steve jobs chết thì những mẫu iphone về...
2807    có cái này tiện hẳn luôn , thích nhất là nhiệt...
4283    thật ra nhà táo tung sản phẩm của mình ra là đ...
4837    cái ip bạn nói k phải of aple nha . k bgio có ...
2875       thích quá . về vn không biết giá bao nhiêu đây
168                                             thất vọng
2186    công nhận vn chuộng iphone không thua bất cứ n...
1036    y chan

In [ ]:
dataset=dataset.drop_duplicates("Data")
dataset = dataset.dropna()

In [ ]:
dataset = dataset.drop(dataset[dataset["Data"].map(len) < 2].index)

In [ ]:
import random
_dataset = {
    "train": [],
    "eval": []
}
temp_dataset = [(data["Data"], data["Class"]) for _, data in dataset.iterrows()]
random.seed(1234)
random.shuffle(temp_dataset)

In [ ]:
split_ratio = {
    "train": 0.8,  # 80% of the data for training
    "eval": 0.2    # 20% of the data for evaluation
}

_i = 0
_len = len(temp_dataset)
for mode in split_ratio.keys():
    end = _i + int(_len * split_ratio[mode])
    _dataset[mode] = temp_dataset[_i:end]
    _i = end

In [ ]:
from collections import Counter
import json

for mode in _dataset.keys():
    print(len(_dataset[mode]))
    a = Counter([e[1] for e in _dataset[mode]])
    print(a)
    json.dump({"data":_dataset[mode]}, open(f"{TRAINING_DATASET_PATH}/training/{mode}.json", "w", encoding="utf8"), indent=4, ensure_ascii=False)

4028
Counter({0: 1369, 1: 1341, 2: 1318})
1007
Counter({1: 355, 2: 332, 0: 320})


In [ ]:
!pip install vncorenlp
!pip install onnx
!pip install onnxruntime
!pip install -U git+https://github.com/huggingface/transformers.git
import onnx
from transformers import RobertaForSequenceClassification
from onnxruntime.transformers.optimizer import optimize_model
from transformers.convert_graph_to_onnx import quantize
import onnxruntime as ort
import torch
from functools import reduce

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-m2u1nhze
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-m2u1nhze
  Resolved https://github.com/huggingface/transformers.git to commit f70db28322150dd986298cc1d1be8bc144cc1a88
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from transformers import AutoModel, AutoTokenizer, RobertaForSequenceClassification
from transformers import Trainer, SchedulerType
from transformers.training_args import TrainingArguments, OptimizerNames
from transformers.trainer_utils import IntervalStrategy
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import py_vncorenlp
import json
import random

In [ ]:
from vncorenlp import VnCoreNLP
class PipelineConfig:
    rdrsegmenter = VnCoreNLP("model/VnCoreNLP-1.2.jar", annotators="wseg")

In [ ]:
PipelineConfig.rdrsegmenter.tokenize("Tôi là một kỹ sư AI")

[['Tôi', 'là', 'một', 'kỹ_sư', 'AI']]

In [ ]:
class SentimentConfig:
    def __init__(
            self,
            pretrained_path: str = None,
            dataset_path: str = None,
            training_output_dir: str = None,
            training_learning_rate: float = None,
            training_weight_decay: float = None,
            training_batch_size: int = None,
            training_num_epochs: int = None,
            training_save_total_limit: int = None,
            training_gradient_accumulation_steps: int = None,
            training_eval_steps: int = None,
            training_logging_steps: int = None,
            training_save_steps: int = None,
            training_logging_dir: str = None,
            training_warm_up_ratio: float = None,
            training_device: str = None,
            training_metrics: str = None,
            segmentor_dir:str = None,
            model_input_max_length:int= None,
            model_num_labels:int=None
    ):
        self.pretrained_path = pretrained_path if pretrained_path is not None else "vinai/phobert-base-v2"
        self.dataset_path = dataset_path if dataset_path is not None else f"{TRAINING_DATASET_PATH}/training"
        self.training_output_dir = training_output_dir if training_output_dir is not None else f"{OUTPUT_DIR}/checkpoint/checkpoint20_1"
        self.training_logging_dir = training_logging_dir if training_logging_dir is not None else f"{OUTPUT_DIR}/logging"
        self.training_learning_rate = training_learning_rate if training_learning_rate is not None else 1e-5
        self.training_weight_decay = training_weight_decay if training_weight_decay is not None else 0.01
        self.training_batch_size = training_batch_size if training_batch_size is not None else 16
        self.training_num_epochs = training_num_epochs if training_num_epochs is not None else 10
        self.training_save_total_limit = training_save_total_limit if training_save_total_limit is not None else 10
        self.training_gradient_accumulation_steps = training_gradient_accumulation_steps if training_gradient_accumulation_steps is not None else 2
        self.training_eval_steps = training_eval_steps if training_eval_steps is not None else 50
        self.training_logging_steps = training_logging_steps if training_logging_steps is not None else 50
        self.training_save_steps = training_save_steps if training_save_steps is not None else 50
        self.training_warm_up_ratio = training_warm_up_ratio if training_warm_up_ratio is not None else 0.05
        self.training_device = training_device if training_device is not None else "cuda"
        self.training_metrics = training_metrics if training_metrics is not None else "f1"
        self.model_input_max_length = model_input_max_length if model_input_max_length is not None else 200
        self.model_num_labels = model_num_labels if model_num_labels is not None else 3

In [ ]:
random.seed(2023)

class SentimentDataset(Dataset):
    def __init__(self, config, mode, tokenizer):
        super().__init__()
        self.config = config
        self.tokenizer = tokenizer
        self.data = self.load_dataset(path=f"{self.config.dataset_path}/{mode}.json")
        random.shuffle(self.data)
        print(Counter([e[1] for e in self.data]))
        print(f"Load {len(self.data)} examples for {mode} dataset")

    def load_dataset(self, path: str):
        data = json.load(open(path, "r", encoding="utf8"))
        return data["data"]

    def __getitem__(self, idx):
        data_item = self.data[idx]
        labels = torch.zeros(self.config.model_num_labels)
        labels[data_item[-1]] = 1
        text = data_item[0].lower()
        segmented_text = PipelineConfig.rdrsegmenter.tokenize(text)
        tokenized_text = self.tokenizer([" ".join(segmented_text[0])], padding="max_length", truncation=True, max_length=self.config.model_input_max_length, return_tensors="pt")
        return {
            **tokenized_text,
            "labels": labels
        }

    def __len__(self):
        return len(self.data)


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
from typing import Optional
from torch.nn import BCEWithLogitsLoss, CrossEntropyLoss, MSELoss
from transformers.modeling_outputs import SequenceClassifierOutput


class SentimentRobertaClassificationHead(nn.Module):
    """Head for sentence-level classification tasks."""

    def __init__(self, config):
        super().__init__()
        self.lstm = nn.LSTM(config.hidden_size, config.hidden_size, 2, bidirectional=True)
        self.dense = nn.Linear(2 * config.hidden_size, config.hidden_size)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.out_proj = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, features, **kwargs):
        # x = features[:, 0, :]  # take <s> token (equiv. to [CLS])
        x = features.mean(dim=1)
        x, (h, c) = self.lstm(x)
        x = self.dropout(x)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x


class SentimentModel(RobertaForSequenceClassification):
  def __init__(self, config):
    super().__init__(config)
    self.classifier = SentimentRobertaClassificationHead(config)

  def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.FloatTensor] = None,
        token_type_ids: Optional[torch.LongTensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        head_mask: Optional[torch.FloatTensor] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ):

      return_dict = return_dict if return_dict is not None else self.config.use_return_dict
      outputs = self.roberta(
          input_ids,
          attention_mask=attention_mask,
          token_type_ids=token_type_ids,
          position_ids=position_ids,
          head_mask=head_mask,
          inputs_embeds=inputs_embeds,
          output_attentions=output_attentions,
          output_hidden_states=output_hidden_states,
          return_dict=return_dict,
      )
      sequence_output = outputs[0]
      logits = self.classifier(sequence_output)

      loss = None
      if labels is not None:
          # move labels to correct device to enable model parallelism
          labels = labels.to(logits.device)
          if self.config.problem_type is None:
              if self.num_labels == 1:
                  self.config.problem_type = "regression"
              elif self.num_labels > 1 and (labels.dtype == torch.long or labels.dtype == torch.int):
                  self.config.problem_type = "single_label_classification"
              else:
                  self.config.problem_type = "multi_label_classification"

          if self.config.problem_type == "regression":
              loss_fct = MSELoss()
              if self.num_labels == 1:
                  loss = loss_fct(logits.squeeze(), labels.squeeze())
              else:
                  loss = loss_fct(logits, labels)
          elif self.config.problem_type == "single_label_classification":
              loss_fct = CrossEntropyLoss()
              loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
          elif self.config.problem_type == "multi_label_classification":
              loss_fct = BCEWithLogitsLoss()
              loss = loss_fct(logits, labels)

      if not return_dict:
          output = (logits,) + outputs[2:]
          return ((loss,) + output) if loss is not None else output

      return SequenceClassifierOutput(
          loss=loss,
          logits=logits,
          hidden_states=outputs.hidden_states,
          attentions=outputs.attentions,
      )

In [ ]:
from evaluate import load
from transformers.trainer_utils import EvalPrediction
from collections import Counter


class SentimentTrainer:
    def __init__(self, config: SentimentConfig = None):
        self.config = config if config is not None else SentimentConfig()
        self.model = SentimentModel.from_pretrained(self.config.pretrained_path, num_labels=self.config.model_num_labels)
        self.model = self.model.float()
        self.model.to(self.config.training_device)
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.pretrained_path)
        self.metrics = {
            "precision": load("precision"),
            "recall": load("recall"),
            "f1": load("f1"),
            "accuracy": load("accuracy")
        }
        self.train_dataset, self.eval_dataset = self.load_dataset()

        self.init_trainer()

    def load_dataset(self):
        train_dataset = SentimentDataset(config=self.config, mode="train", tokenizer=self.tokenizer)
        eval_dataset = SentimentDataset(config=self.config, mode="eval", tokenizer=self.tokenizer)
        return train_dataset, eval_dataset

    def collator(self, data):
        batch = {}
        _keys = data[0].keys()
        for key in _keys:
            batch[key] = torch.vstack([ele[key] for ele in data]).to(self.config.training_device)

        return batch

    def compute_metrics(self, eval_predictions: EvalPrediction):
        predictions, labels = eval_predictions
        predictions = torch.from_numpy(predictions).softmax(dim=-1).argmax(dim=-1)
        labels = torch.from_numpy(labels).argmax(dim=-1)

        output =  {
            "precision": self.metrics["precision"].compute(predictions=predictions, references=labels,
                                                           average="weighted")["precision"],
            "recall": self.metrics["recall"].compute(predictions=predictions, references=labels, average="weighted")["recall"],
            "f1": self.metrics["f1"].compute(predictions=predictions, references=labels, average="weighted")["f1"],
            "accuracy": self.metrics["accuracy"].compute(predictions=predictions, references=labels)["accuracy"]
        }
        return output

    def init_trainer(self):
        training_args = TrainingArguments(
            output_dir=self.config.training_output_dir,
            evaluation_strategy=IntervalStrategy.STEPS,
            save_strategy=IntervalStrategy.STEPS,
            per_device_train_batch_size=self.config.training_batch_size,
            per_device_eval_batch_size=self.config.training_batch_size,
            learning_rate=self.config.training_learning_rate,
            weight_decay=self.config.training_weight_decay,
            save_total_limit=self.config.training_save_total_limit,
            num_train_epochs=self.config.training_num_epochs,
            gradient_accumulation_steps=2,
            eval_steps=self.config.training_eval_steps,
            fp16=self.config.training_device == "cuda",
            push_to_hub=False,
            dataloader_num_workers=0,
            logging_dir=self.config.training_logging_dir,
            logging_strategy=IntervalStrategy.STEPS,
            logging_steps=self.config.training_logging_steps,
            save_steps=self.config.training_save_steps,
            logging_first_step=False,
            optim=OptimizerNames.ADAMW_TORCH,
            load_best_model_at_end=True,
            metric_for_best_model=self.config.training_metrics,
            warmup_ratio=self.config.training_warm_up_ratio,
            lr_scheduler_type=SchedulerType.COSINE_WITH_RESTARTS,
            dataloader_pin_memory=False
        )
        self.trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=self.train_dataset,
            eval_dataset=self.eval_dataset,
            tokenizer=self.tokenizer,
            data_collator=self.collator,
            compute_metrics=self.compute_metrics
        )

    def train(self):
        self.trainer.train()

    def eval(self):
        print(self.trainer.evaluate(eval_dataset=self.eval_dataset))

In [ ]:
trainer = SentimentTrainer()

Some weights of SentimentModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.lstm.bias_hh_l1_reverse', 'classifier.out_proj.bias', 'classifier.lstm.bias_ih_l0', 'classifier.lstm.weight_ih_l0_reverse', 'classifier.lstm.weight_hh_l0_reverse', 'classifier.lstm.weight_ih_l1_reverse', 'classifier.lstm.bias_hh_l1', 'classifier.lstm.weight_hh_l1', 'classifier.lstm.weight_ih_l1', 'classifier.lstm.bias_ih_l1', 'classifier.lstm.bias_hh_l0', 'classifier.lstm.weight_hh_l1_reverse', 'classifier.lstm.weight_ih_l0', 'classifier.lstm.bias_ih_l0_reverse', 'classifier.lstm.bias_hh_l0_reverse', 'classifier.lstm.weight_hh_l0', 'classifier.dense.bias', 'classifier.lstm.bias_ih_l1_reverse', 'classifier.out_proj.weight', 'classifier.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5.

Counter({0: 1369, 1: 1341, 2: 1318})
Load 4028 examples for train dataset
Counter({1: 355, 2: 332, 0: 320})
Load 1007 examples for eval dataset


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
50,0.682400,0.656380,0.441638,0.382324,0.309005,0.382324
100,0.640900,0.638077,0.211604,0.325720,0.193526,0.325720
150,0.638200,0.637409,0.579558,0.342602,0.241726,0.342602
200,0.636600,0.633731,0.360749,0.463754,0.368330,0.463754
250,0.623100,0.611624,0.537211,0.500497,0.465276,0.500497
300,0.594600,0.580077,0.545409,0.551142,0.499169,0.551142
350,0.549000,0.538223,0.611530,0.613704,0.598715,0.613704
400,0.517400,0.513331,0.636816,0.630586,0.623883,0.630586
450,0.481200,0.497214,0.670162,0.647468,0.617116,0.647468
500,0.463100,0.476585,0.697219,0.680238,0.653537,0.680238


Removed shared tensor {'classifier.lstm.weight_hh_l0_reverse', 'classifier.lstm.weight_ih_l1_reverse', 'classifier.lstm.bias_hh_l0_reverse', 'classifier.lstm.weight_hh_l0', 'classifier.lstm.bias_hh_l1_reverse', 'classifier.lstm.bias_hh_l1', 'classifier.lstm.weight_hh_l1', 'classifier.lstm.weight_ih_l1', 'classifier.lstm.bias_ih_l1', 'classifier.lstm.bias_ih_l1_reverse', 'classifier.lstm.weight_ih_l0_reverse', 'classifier.lstm.bias_hh_l0', 'classifier.lstm.weight_hh_l1_reverse', 'classifier.lstm.bias_ih_l0', 'classifier.lstm.bias_ih_l0_reverse'} while saving. This should be OK, but check by verifying that you don't receive any warning while reloading
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/skl

In [ ]:
best_checkpoint_path = f"{trainer.config.training_output_dir}/checkpoint-1250"
best_model = RobertaForSequenceClassification.from_pretrained(best_checkpoint_path, ignore_mismatched_sizes=True)
best_model.eval()
tokenizer = AutoTokenizer.from_pretrained(best_checkpoint_path)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at /content/output/checkpoint/checkpoint20_1/checkpoint-1250 and are newly initialized because the shapes did not match:
- classifier.dense.weight: found shape torch.Size([768, 1536]) in the checkpoint and torch.Size([768, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**Test**

In [ ]:
import scipy

In [ ]:
TEST_PATH = f"drive/MyDrive/vlsp_sentiment_test.csv"
test_dataset = pd.read_csv(TEST_PATH, sep='\t')

In [ ]:
test_dataset

,Class,Data
0,-1,Nói thiệt là mình thì thì chuột nào mình cũng ...
1,-1,Đang dùng mx1. Cũng ngon nhưng chưa đầy năm mà...
2,-1,"Chưa thấy đc điểm thuyết phục để mua, nhất là ..."
3,-1,"Những phần xem báo tra cứu bản đồ, dịch vụ.. d..."
4,-1,ĐÚNG LÀ MUA Ở VIỆT NAM KHÔNG ỨNG DỤNG ĐƯỢC GÌ ...
...,...,...
1045,0,30 củ à :)
1046,0,Apple bán dc thi samsung cũng lời nhiêu. Ng...
1047,0,có thể giúp android vượt trội so với ios chớ c...
1048,0,Mẹ mình từng sang Đài Loan và có mua 1 cái iph...


In [ ]:
# Define a mapping dictionary
class_mapping = {-1: 0, 0: 1, 1: 2}

# Apply the mapping to the 'Class' column
test_dataset['Class'] = test_dataset['Class'].map(class_mapping)

In [ ]:
test_dataset

,Class,Data
0,0,Nói thiệt là mình thì thì chuột nào mình cũng ...
1,0,Đang dùng mx1. Cũng ngon nhưng chưa đầy năm mà...
2,0,"Chưa thấy đc điểm thuyết phục để mua, nhất là ..."
3,0,"Những phần xem báo tra cứu bản đồ, dịch vụ.. d..."
4,0,ĐÚNG LÀ MUA Ở VIỆT NAM KHÔNG ỨNG DỤNG ĐƯỢC GÌ ...
...,...,...
1045,1,30 củ à :)
1046,1,Apple bán dc thi samsung cũng lời nhiêu. Ng...
1047,1,có thể giúp android vượt trội so với ios chớ c...
1048,1,Mẹ mình từng sang Đài Loan và có mua 1 cái iph...


In [ ]:
temp_dataset = [(data["Data"], data["Class"]) for _, data in test_dataset.iterrows()]

In [ ]:
_dataset = {
    "test" : []
}

split_ratio = {
    "test": 1
}

In [ ]:
_i = 0
_len = len(temp_dataset)
_dataset["test"] = temp_dataset[_i:int(_len*split_ratio["test"])+1]

In [ ]:
CURRENT_PATH

'/content'

In [ ]:
!mkdir dataset/testing
print(len(_dataset["test"]))
a = Counter([e[1] for e in _dataset["test"]])
print(a)
json.dump({"data":_dataset["test"]}, open(f"{CURRENT_PATH}/dataset/testing/test.json", "w", encoding="utf8"), indent=4, ensure_ascii=False)

mkdir: cannot create directory ‘dataset/testing’: File exists
1050
Counter({0: 350, 2: 350, 1: 350})


In [ ]:
def reverse_label(label):
    label_map = {
        0: "NEG",
        1: "NEU",
        2: "POS"
    }
    return label_map[label]

In [ ]:
import json
test_dataset_json = json.load(open(f"{CURRENT_PATH}/dataset/testing/test.json", "r", encoding="utf8"))

In [ ]:
_len = len(_dataset["test"])
subset = random.choices(test_dataset_json["data"], k=_len)
acc=0
for example, label in subset:
    segmented_text = [" ".join(PipelineConfig.rdrsegmenter.annotate(example))]
    _inputs = tokenizer(segmented_text, return_tensors="pt", padding=True)

    # Feed the inputs to the model
    model_output = best_model(**_inputs)

    # Detach the tensor and convert it to numpy
    logits = model_output.logits.detach().numpy()

    # Get the probabilities by applying softmax
    probabilities = scipy.special.softmax(logits, axis=-1)

    # Get the predicted label
    pred = probabilities.argmax(-1).tolist()[0]

    if label == pred:
        acc += 1
    else:
        print("==============")
        print(example)
        print(reverse_label(label), "!=", reverse_label(pred))

# Calculate accuracy
accuracy = acc / _len
print(f"Accuracy: {accuracy}")

8gb là đủ rồi, còn card thì lấy asus 4gb 4m hoặc gtx950 4m tuỳ sở thích
NEU != POS
note 7 5.7 icnch ma be hon nhieu so voi iphone 6s plus co man 5.5 inch nhi. Co ve apple thiet ke iphone man to ko can doi
NEU != POS
Hàng chính hãng giá khoảng từ 16 triệu. Hàng của bạn định mua là hàng 10 hãng giá từ khoảng 500 tệ. Bạn có thể mua xài để có chút kinh nghiệm và không bị lầm lần sau. Chúc bạn tìm được điện thoại như ý !
NEU != POS
Nút home cảm ứng ở trong màn hình sẽ là một thất bại tiếp theo của Sony
NEG != POS
Mua về vài năm nó hỏng thì chát nhỉ. Mà các bác cho e hỏi. Mua ssd về chỉ nghi vào thôi k sóa đi thì để lâu có hỏng k nhỉ.
NEU != POS
hang nao ma cha la hang chinh hang . hong le mang tu my zia la hang khong chinh hang a may thanh
NEU != POS
Apple không nên chạy theo mẫu HTC, quá xấu. Cứ để thiết kế theo kết cấu phom như 6s Plus và 5s và 5SE là ok nhất.
NEG != POS
Đã từng dùng con Neo7, ram gì có 1GB, dùng gì mà chán kinh. tưởng đâu đt giá rẻ bị vậy, bữa cầm con Zenfone Laser của t